# UrbanSound8K CNN and Transformer 10-Fold Cross Validation on Google Colab

Purpose: run formal 10-fold cross validation for both the CNN baseline and the Spectrogram Transformer on Colab GPU.

Storage policy:
- GitHub stores code, configs, notebooks, and documentation.
- Google Drive stores the large UrbanSound8K dataset and processed Mel-spectrogram cache.
- Colab is only the temporary GPU runtime.

Before running: open `Runtime > Change runtime type` and select a GPU accelerator.

In [ ]:
# Mount Google Drive so the dataset and processed spectrograms persist after Colab restarts.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check the GPU. A Tesla T4 or similar GPU is enough for this MVP training run.
!nvidia-smi

In [ ]:
# Clone the project code from GitHub into the temporary Colab filesystem.
# If the repository already exists in this runtime, this cell enters it instead of cloning again.
%cd /content
!test -d urbansound8k-thesis || git clone https://github.com/lezhe0414/urbansound8k-thesis.git
%cd /content/urbansound8k-thesis

# Record the exact code version used for the experiment.
!git log --oneline -1

In [ ]:
# Install dependencies required by the training and preprocessing scripts.
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

## Dataset Cache

Run the next cell only if the dataset has not already been downloaded to Google Drive. After it succeeds once, comment out `dataset.download()` and keep `dataset.validate()`.

In [ ]:
# Download UrbanSound8K to Google Drive and validate file completeness.
# After the first successful download, comment out dataset.download().
import soundata

dataset = soundata.initialize(
    "urbansound8k",
    data_home="/content/drive/MyDrive/urbansound8k_data/UrbanSound8K_soundata",
)

dataset.download()
dataset.validate()

## Spectrogram Cache

Run preprocessing once after the dataset is available. It converts raw audio into Mel-spectrogram `.npz` files stored in Google Drive. After it succeeds once, you can skip this cell in later Colab sessions.

In [ ]:
# Convert raw UrbanSound8K audio into normalized Mel-spectrogram features.
!python -m src.preprocess \
  --raw-dir /content/drive/MyDrive/urbansound8k_data/UrbanSound8K_soundata \
  --out-dir /content/drive/MyDrive/urbansound8k_data/urbansound8k_mels

In [ ]:
# Link the Drive spectrogram cache to the path expected by the training configs.
# This cell must be run after every Colab runtime restart.
!mkdir -p data/processed
!rm -rf data/processed/urbansound8k_mels
!ln -s /content/drive/MyDrive/urbansound8k_data/urbansound8k_mels data/processed/urbansound8k_mels
!ls -la data/processed

## CNN Baseline 10-Fold Cross Validation

This is the thesis baseline model. It treats each Mel-spectrogram as a single-channel image.

In [ ]:
# Train the CNN baseline across all official UrbanSound8K folds.
# This writes per-fold outputs and a 10-fold summary under results/.
!python -m src.train --config configs/cnn_baseline.yaml --fold all

In [ ]:
# Inspect the CNN 10-fold summary.
!cat results/cnn_baseline_10fold_summary.json

In [ ]:
# Print the CNN 10-fold mean metrics in a compact format.
import json
from pathlib import Path

metrics_path = Path("results/cnn_baseline_10fold_summary.json")
with metrics_path.open("r", encoding="utf-8") as f:
    summary = json.load(f)

print("CNN baseline 10-fold mean metrics")
for key, value in summary["mean"].items():
    print(f"{key}: {value}")

## Spectrogram Transformer 10-Fold Cross Validation

This is the modern comparison model. It splits each Mel-spectrogram into patches and classifies the sound event with a Transformer encoder.

In [ ]:
# Train the Spectrogram Transformer across all official UrbanSound8K folds.
# This writes per-fold outputs and a 10-fold summary under results/.
!python -m src.train --config configs/transformer_baseline.yaml --fold all

In [ ]:
# Inspect the Transformer 10-fold summary.
!cat results/transformer_baseline_10fold_summary.json

In [ ]:
# Print the Transformer 10-fold mean metrics in a compact format.
import json
from pathlib import Path

metrics_path = Path("results/transformer_baseline_10fold_summary.json")
with metrics_path.open("r", encoding="utf-8") as f:
    summary = json.load(f)

print("Spectrogram Transformer 10-fold mean metrics")
for key, value in summary["mean"].items():
    print(f"{key}: {value}")

## Package Results

Run this after CNN and/or Transformer 10-fold training finishes. It packages result folders, summary files, and confusion matrix figures into one zip file for download back to the local repository.

In [ ]:
# Package all 10-fold cross-validation artifacts generated in this Colab session.
!zip -r 10fold_model_comparison_artifacts.zip \
  results/cnn_baseline_fold* \
  results/transformer_baseline_fold* \
  results/cnn_baseline_10fold_summary.json \
  results/cnn_baseline_10fold_summary.csv \
  results/transformer_baseline_10fold_summary.json \
  results/transformer_baseline_10fold_summary.csv \
  figures/cnn_baseline_fold*_confusion_matrix.png \
  figures/transformer_baseline_fold*_confusion_matrix.png

## Local Sync Checklist

After Colab finishes:

1. Download `10fold_model_comparison_artifacts.zip`.
2. Unzip it into the local repository root.
3. Update `docs/progress_tracker.md` with CNN and Transformer metrics.
4. Run `python3 scripts/check_project_status.py` locally.
5. Commit only code/docs/config changes. Do not commit raw data or large generated artifacts unless explicitly required.